# SemFlow / NganyaFlow waiting-time pipeline
Synthetic validation only. Target: residual minutes until next arrival. No Nairobi field-accuracy claim. Run with the installed waiting_time package (Python 3.12+). Earlier MTA notebooks are preserved background, not this target.

In [1]:
import sys
from pathlib import Path
import json
from nganyaflow.prepare_waiting import prepare
from nganyaflow.schemas import Catalog
from nganyaflow.ml import train
root = Path.cwd()
if root.name == "notebooks": root = root.parent
sys.path.insert(0, str(root / "waiting_time/src"))
bundle = json.loads((root / "waiting_time/data/examples/nairobi-cbd-synthetic-v1.json").read_text())
catalog = Catalog.model_validate_json((root / "waiting_time/data/reference/catalog.json").read_text())
arrivals, queries = prepare(bundle, catalog)
print(bundle["metadata"])
print({"arrivals": len(arrivals), "labelled_queries": len(queries)})

{'dataset_id': 'nairobi-cbd-synthetic-v1', 'data_kind': 'synthetic', 'seed': 20261027, 'generator_version': '1.0', 'generated_for': 'AI coursework validation', 'description': 'Real Nairobi CBD place names; all arrival times and route connectivity simulated.', 'period': '2026-10-05 to 2026-10-18', 'license': 'Project-generated teaching data', 'location_sources': ['https://africa.itdp.org/wp-content/uploads/2015/02/Nairobi-Ndovu-A104-BRT-Service-Plan.pdf', 'https://krc.co.ke/nairobi-railway-city-3/'], 'assumptions': {'session_minutes': 90, 'hours_local': [7, 12], 'base_headways_minutes': {'kencom': 4.5, 'odeon': 6.0, 'railways': 8.0}, 'midday_multiplier': 1.55, 'weekend_multiplier': 1.2, 'innovation_std_minutes': 1.2, 'headway_persistence': 0.35, 'note': 'Parameters illustrate learnable relationships, not Nairobi measurements.'}}
{'arrivals': 1049, 'labelled_queries': 5943}


## Chronological evaluation
Earlier complete dates train; the last three dates test. No random minute-row split. The future arrival supplies the label only. This writes research artifacts and never activates a service model. MAE is minutes, not classification accuracy.

In [2]:
report = train(arrivals, root / "reports/waiting-time/model.joblib", test_days=3)
report

{'model_version': 'c019e4dcbd28bb25',
 'data_kind': 'synthetic',
 'dataset_ids': ['nairobi-cbd-synthetic-v1'],
 'data_sha256': 'e9dd68a64621b13d54f4eb75c964130c6bbe9317a8000dee0ae1acf474b9daea',
 'configuration': {'min_samples': 30,
  'min_days': 3,
  'test_days': 3,
  'n_estimators': 100,
  'min_samples_leaf': 5,
  'max_features': 1.0,
  'max_samples_leaf': 'all',
  'random_state': 42,
  'label_grid_minutes': 1,
  'maximum_interval_minutes': 120},
 'software': {'python': '3.12.10',
  'quantile-forest': '1.4.2',
  'scikit-learn': '1.9.1',
  'numpy': '2.5.4'},
 'input_arrivals': 1049,
 'input_sessions': 84,
 'train_sessions': 66,
 'test_sessions': 18,
 'train_days': ['2026-10-05',
  '2026-10-06',
  '2026-10-07',
  '2026-10-08',
  '2026-10-09',
  '2026-10-10',
  '2026-10-11',
  '2026-10-12',
  '2026-10-13',
  '2026-10-14',
  '2026-10-15'],
 'test_days': ['2026-10-16', '2026-10-17', '2026-10-18'],
 'train_samples': 4684,
 'test_samples': 1259,
 'excluded_test_samples': 0,
 'mae_minutes': 